# 🔧 Exhibition Connector RAG1 — Fully Fixed for Colab

همه باگ‌ها رفع شده:
✅ تعارض `requests` با `google-colab`
✅ خطای `TypeError` در توکن
✅ `UnstructuredExcelLoader` → `openpyxl`
✅ `FAISS.load_local` + `allow_dangerous_deserialization`
✅ `invoke()` به جای `get_relevant_documents()`
✅ `langchain_text_splitters` به جای `langchain.text_splitter`

In [ ]:
# سلول ۱: نصب کتابخانه‌ها
!pip install -q gradio langchain langchain-community langchain-text-splitters faiss-cpu beautifulsoup4 openpyxl transformers sentence-transformers accelerate
!pip install -q requests==2.32.4

In [ ]:
# سلول ۲: توکن HuggingFace
import os
from getpass import getpass

HF_TOKEN = None

try:
    from google.colab import userdata
    _token = userdata.get('HF_TOKEN')
    if isinstance(_token, str) and _token.strip():
        HF_TOKEN = _token
        print("✅ توکن از Colab Secrets خوانده شد")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = getpass("🔑 توکن HuggingFace خود را وارد کنید: ")

os.environ["HF_TOKEN"] = str(HF_TOKEN)
print("✅ توکن تنظیم شد")

In [ ]:
# سلول ۳: دانلود و بارگذاری اسناد
import requests as req
from pathlib import Path
import openpyxl
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)

xls_path = DATA_DIR / "iran-oil_iran-oil_iran_oil.xlsx"
xls_url = "https://huggingface.co/spaces/sosa123454321/Exhibition-connector-rag1/resolve/main/iran-oil_iran-oil_iran%20oil.xlsx"

if not xls_path.exists():
    print("⏳ دانلود فایل اکسل...")
    r = req.get(xls_url, timeout=60)
    r.raise_for_status()
    xls_path.write_bytes(r.content)
    print(f"✅ دانلود شد ({len(r.content):,} بایت)")

print("⏳ بارگذاری ویکی‌پدیا...")
wiki_docs = WebBaseLoader('https://fa.wikipedia.org/wiki/اوهیا').load()
print(f"✅ ویکی‌پدیا: {len(wiki_docs)} سند")

print("⏳ بارگذاری اکسل با openpyxl...")
wb = openpyxl.load_workbook(str(xls_path))
excel_docs = []
for sn in wb.sheetnames:
    ws = wb[sn]
    rows = ["\t".join(str(c) if c else "" for c in row) for row in ws.iter_rows(values_only=True)]
    excel_docs.append(Document(page_content="\n".join(rows), metadata={"source": str(xls_path), "sheet": sn}))
    print(f"✅ اکسل: شیت '{sn}' → {ws.max_row} ردیف")

all_docs = wiki_docs + excel_docs
print(f"📄 مجموع: {len(all_docs)} سند")

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = splitter.split_documents(all_docs)
print(f"✂️ {len(splits)} چانک")

In [ ]:
# سلول ۴: ساخت ایندکس FAISS
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⏳ ساخت FAISS روی {device}...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = FAISS.from_documents(splits, embeddings)
vectorstore.save_local(str(DATA_DIR / "faiss_index"))
retriever = vectorstore.as_retriever()
print(f"✅ FAISS ساخته شد ({vectorstore.index.ntotal} بردار)")

for q in ["نمایشگاه نفت", "شرکت پتروشیمی"]:
    docs = retriever.invoke(q)
    print(f"  🔍 '{q}' → {len(docs)} نتیجه")

In [ ]:
# سلول ۵: بارگذاری مدل LLM
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

if torch.cuda.is_available():
    mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"🖥️ GPU: {torch.cuda.get_device_name(0)} ({mem:.0f} GB)")
    if mem >= 15:
        MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
        NEED_TOKEN = True
    else:
        MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
        NEED_TOKEN = False
else:
    MODEL = "gpt2"
    NEED_TOKEN = False
    print("⚠️ بدون GPU — فقط تست")

print(f"📦 مدل: {MODEL}")
tok = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN if NEED_TOKEN else None)
mdl = AutoModelForCausalLM.from_pretrained(
    MODEL,
    token=HF_TOKEN if NEED_TOKEN else None,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
llm_pipe = pipeline("text-generation", model=mdl, tokenizer=tok, max_new_tokens=512, temperature=0.8, repetition_penalty=1.1)
print("✅ مدل بارگذاری شد")

In [ ]:
# سلول ۶: تعریف و تست RAG
def ask(question):
    if not question.strip():
        return "لطفاً سوال وارد کنید."
    try:
        docs = retriever.invoke(question)
        ctx = "\n\n".join(d.page_content for d in docs)[:2000]
        prompt = f"شما یک دستیار هوشمند هستید که به زبان فارسی پاسخ می‌دهید.\nسوال: {question}\n\nمتن مرتبط:\n{ctx}\n\nلطفاً پاسخ دقیق و کامل به زبان فارسی بدهید."
        out = llm_pipe(prompt)[0]["generated_text"]
        return out[len(prompt):].strip()
    except Exception as e:
        return f"خطا: {e}"

for q in ["اطلاعات نفت ایران چیست؟", "شرکت‌های پتروشیمی کدامند؟"]:
    print(f"\n❓ {q}")
    print(f"🤖 {ask(q)}")

### Creating a Formal LangChain RAG Chain
We will now use `LangChain` to create a more robust chain that links the `retriever` and the `LLM` together.

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Create a LangChain wrapper for our pipeline
hf_llm = HuggingFacePipeline(pipeline=llm_pipe)

# 2. Define the template
template = """
شما یک دستیار هوشمند هستید.
با استفاده از متن ارائه شده به سوال پاسخ دهید. اگر پاسخ در متن نیست، بگویید که نمی‌دانید.

متن مرتبط:
{context}

سوال: {question}

پاسخ:"""

prompt = PromptTemplate.from_template(template)

# 3. Build the RAG Chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | hf_llm
    | StrOutputParser()
)

print("✅ RAG Chain created successfully.")

# Quick test of the new chain
test_question = "در مورد نمایشگاه نفت توضیح دهید."
print(f"\nTesting new chain with: {test_question}")
response = rag_chain.invoke(test_question)
print(f"Response: {response}")

In [ ]:
# سلول ۷: رابط Gradio
import gradio as gr

def rag_ask(question):
    if not question.strip():
        return "لطفاً سوال وارد کنید."
    try:
        # Using the rag_chain created in the previous cell
        return rag_chain.invoke(question)
    except Exception as e:
        return f"خطا: {e}"

iface = gr.Interface(
    fn=rag_ask,
    inputs=gr.Textbox(lines=2, placeholder="سوال خود را اینجا وارد کنید..."),
    outputs="text",
    title="هوش مصنوعی پاسخگو به سوالات نمایشگاه (نسخه LCEL)",
    description="سلام! با استفاده از زنجیره LangChain، هر سوالی درباره شرکت‌ها یا نمایشگاه دارید بپرسید.",
)
iface.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5f3959a46bfecd0ab5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Okay, I will now execute all cells in the notebook. Please let me know if you encounter any errors or if you'd like me to debug specific parts.